# Designing a Web Crawler

Let's design a Web Crawler that will browse and download the World Wide Web. 

## What's a Web Crawler?
It's a software program which browses the WWW in a methodical and automated manner, collecting documents by recursively fetching links from a set of starting pages.

Search engines use web crawling as a means to provide up-to-date data. Search engines download all pages and create an index on them to perform faster searches.

Other uses of web crawlers:
- Test web pages and links for valid syntax and structure.
- To search for copyright infringements.
- To maintain mirror sites for popular web sites.
- To monitor sites to see where their content changes.

## 1. Requirements and Goals of the System
**Scalability:** Our service needs to be scalable, since we'll be fetching hundreds of millions of web documents.

**Extensibility:** Our service should be designed in a way that allows newer functionality to be added to it. It should be able to allow for newer document formats that needs to be downloaded and processed in future.

## 2. Design Considerations
We should be asking a few questions here:

#### Is it a crawler for only HTML pages? Or should we fetch and store other media types like images, videos, etc.
It's important to clarify this because it will change the design. If we're writing a general-purpose crawler, we might want to break down the parsing module into different sets of modules: one for HTML, another for videos,..., so basically each module handling a given media type.

For this design, let's assume our web crawler will deal with HTML only.

#### What protocols apart from HTTP are we looking at? FTP?
Let's assume HTTP for now. Again, it should not be hard to extend it to other protocols.

#### What is the expected number of pages we will crawl? How big will be the URL Database?
Let's assume we need to crawl 1Billion websites. Since one site can contain many URLs, assume an upper bound of `15 billion web pages`.


#### Robots Exclusion Protocol?
Some web crawlers implement the Robots Exclusion Protocol, which allows Webmasters to declare parts of their sites off limits to crawlers. The rules are published in `robots.txt`, not `robot.txt`.


## 3. Capacity Estimation and Constraints
If we crawl 15B pages in 4 weeks, how many pages will we need to fetch per second?

```text
        15B / (4 weeks * 7 days * 86400 sec) ~= 6200 web pages/sec
```

**What about storage?** Pages sizes vary. But since we are dealing with HTML only, let's assume an average page size is 100KB. With each page, if we're storing 500 bytes of metadata, total storage we would need is:

```text
        15B * (100KB + 500 bytes)
        15 B * 100.5 KB ~= 1.5 Petabytes
```

We don't want to go beyond 70% capacity of our storage system, so the total storage we will need is:

```text
        1.5 petabytes / 0.7 ==> 2.14 Petabytes     
```

##  4. High Level Design
The basic algorithm of a web crawler is this:

1. Taking in a list of seed URLs as input, pick a URL from the unvisited URL list.
2. Find the URL host-name's IP address.
3. Establish a connection to the host to download its corresponding documents.
4. Parse the documents contents to look for new URLs.
5. Add the new URLs to the list of unvisited URLs.
6. Process the downloaded document, e.g, store it, or index the contents
7. Go back to step 1.

### How to Crawl

Breath first or depth first?
Breadth-first search (BFS) is usually used. We can also use Depth-first search especially when the crawler has already established a connection with a website. In this situation, the crawler will just DFS all the URLs within the website to save some handshaking overhead.

**Path-ascending crawling:** Path-ascending crawling helps discover a hidden or isolated resources. In this scheme, a crawler would ascend to every path in each URL like so:
```text
    given a seed URL of http://xyz.com/a/b/one.html

    it will attempt to crawl /a/b/, /a/ and /
```

### Difficulties implementing an efficient web crawler.
#### 1. Large volume of web pages
A large volume implies that the web crawler can only dowload a fraction of the web pages, so it's critical that the web crawler should be intelligent enough to prioritize download.

#### 2. Rate of change on web pages
Web pages change frequenty. By the time the crawler is downloading the last page from the site, the page may change dynamically, or a new page may be added.

**Components of a bare minimum crawler:**
1. **URL frontier:** stores a list of URLs to download and prioritize which URLs should be crawled first.
2. **HTTP Fetcher:** to retrieve a web page from the hosts server.
3. **Extractor:** to extract links from HTML documents.
4. **Duplicate Remover:** to make sure same content is not extracted twice.
5. **Datastore:** to store retrieved pages, URLs and other metadata.

![](images/designing_webcrawler_high_level.png)

## 5. Detailed Component Design
Assume the crawler is running on a single server, where multiple working threads are performing all the steps needed to download and process a document in a loop.

**Step 1:** remove an absolute URL from the shared URL frontier for downloading. the URL begins with a scheme (e.g HTTP) which identifies the network protocol that should be used to download it.
We can implement these protocols in a modular way for extensibility, so that later if our crawler needs to support more protocols, it can easily be done.

**Step 2:** Based on the URL's scheme, the worker calls the appropriate protocol module to download the document.

**Step 3:** After downloading, the document is written into a Document Input Stream (DIS). This will enable other modules to re-read the document multiple times.

**Step 4:** The worker tests whether the document bytes have already been stored. Reuse an existing content object only after confirming equality. Do not skip URL-specific link extraction: identical HTML containing `next.html` resolves differently at `/a/index.html` and `/b/index.html`. Content identity and discovery context are separate.

**Step 5:** Process the downloaded document. Each doc has a different MIME type like HTML page, Image, Video etc. We can implement these MIME schemes in a modular way, to allow for extensibility when our crawler need to support more types. The worker invokes the process method of each processing module with that MIME type.

**Step 6:** The HTML processing module will extract links from the page. Each link is converted into an absolute URL and testsed against a user-supplied filter to determine if it should be downloaded. If the URL passes the filter, the worker performs the URL-dedupe test, which checks if the URL has been downloaded before. If it's new, it is added into the URL frontier.

![](images/designing_crawler_detailed_component.png)

Let's see how each component can be distributed onto multiple machines:

#### a. The URL frontier
This is the data structure that contains all the URLs that are queued to be downloaded. We crawl by performing a breadth-first traversal of the Web, starting from the pages in the seed set. We can use a FIFO queue to implement this.

Since we have a huge list of URLs to crawl, we can distribute our URL frontier into multiple servers. Let's assume on each server we have multiple threads performing the crawling tasks. Our hash function maps each URL to the server responsible for crawling it.

Constraints for a distributed URL frontier:
- The crawler should not overload a server by downloading a lot of pages.
- Multiple machines should not connect to a single web server.

> For each server, we can have distinct FIFO sub-queues, where each worker thread will remove URLs for crawling.

Once a new URL is added, we determine which sub-queue it belongs to by using the URL's canonical hostname. Our hash function will map each **hostname** to a **thread number**. Together, these two points imply, that only one worker thread will download documents from a given web server and also, by using FIFO queue, it'll not overload a web server.

##### How big will our URL frontier be?
The size would be in the 100s of millions of URLs. Therefore, we need to store the URLs on disk. We can implement our queues in such a way that they have separate buffers for enqueuing and dequeuing. Enqueuing buffer, once filled, will be dumped to the disk. Dequeuing buffers will keep a cache of URLs that need to be visited; periodically reading from the disk to fill the buffer.

#### b. The Fetcher Module
This will download the document corresponding to a given URL using the appropriate network protocol like HTTP. Webmasters publish `robots.txt` to declare crawler access rules.
To avoid downloading this text file on every request, our HTTP protocol module can maintain a cache mapping host-names to their robot's exclusion rules.

#### c. Document input steam
We cache the document locally using DIS to avoid downloading the document multiple times.

A DIS is an input stream that caches the doc's entire contents in memory. It also provides methods to re-read the document. Larger documents can be temporarily written to a backing file.

Each worker will have a DIS, which it reuses from document to document. After extracting a URL from the frontier, the worker passes that URL to the relevant protocol module (in our case, for HTTP) which initializes the DIS from a network connection to contain the document's contents. The worker then passes the DIS to all relevant processing modules.

#### d. Document Dedupe test
Content deduplication shares stored page bytes; it does not establish that all processing for a URL has already occurred.

Define equality first. Here, duplicates have identical fetched body bytes; stripping markup or normalizing text would define a different equivalence relation. Retain response metadata and URL-specific processing context separately.

Use an indexed SHA-256 digest to locate candidate content objects, then compare their stored bytes before reusing one. A digest match is a candidate, not proof of equality. MD5 produces 128 bits and SHA-256 produces 256 bits; a 64-bit value would be a truncation, not their full digest. [Python's hash documentation](https://docs.python.org/3/library/hashlib.html) also notes known collision weaknesses in MD5 and SHA-1.

For `n` independent distinct inputs and an ideal uniform `b`-bit hash:

```text
expected colliding pairs = n * (n - 1) / (2 * 2^b)
P(at least one collision) ~= 1 - exp(-expected colliding pairs)
n = 15,000,000,000; b = 64
expected colliding pairs ~= 6.10
P(at least one collision) ~= 99.78%
```

This is a population-wide probability, not a 99.78% collision probability for each document. The uniform-hash model is not a guarantee against deliberately chosen inputs. A wider digest greatly reduces accidental collision risk, but byte comparison is what makes this design's equality decision exact.

##### How big will be the checksum store
For 15 billion distinct content objects, full SHA-256 digests alone occupy:
```text
        15,000,000,000 * 32 bytes = 480,000,000,000 bytes = 480 GB
```
This uses decimal GB and excludes content references, collision candidates, index overhead, replicas, and the page bytes themselves. A 64-bit candidate index would use 120 GB of digest payload, but would still need collision resolution for exact equality.

An LRU cache can reduce candidate lookups, with persistent storage remaining authoritative.

Steps:
- Look up the digest in the cache, falling back to the persistent index on a miss.
- Compare the fetched bytes with candidate objects. A match permits sharing stored bytes, not skipping base-URL-specific link extraction.
- If no candidate is equal, retain a separate content object and add it to the digest's candidate set. Never overwrite another object merely because its digest matches.
- Concurrent publication needs an atomic candidate-index update or transaction so one writer cannot lose another writer's reference. An LRU entry is not that authority.

#### e. URL filters
URL filtering mechanism provides a customizable way to control the set of URLs that are downloaded. We can define filters to restrict URLs by domain, prefix or protocol type.

Before adding the URL to the frontier, the worker thread consults the user-supplied URL filter.

#### f. Domain name resolution
Before we contact a web server, a Web crawler must use a DNS to map the Web server's hostname into  an IP address. DNS name resolution will be a big bottleneck of our crawlers given the amount of URLs we are working with. To avoid repeated requests, we can start caching DNS results by building our local DNS server.

#### g. URL dedupe test
While extracting links, we will encounter multiple links to the same document. To avoid downloading and processing a doc multiple times, we use a URL dedupe test on each extracted link before adding it to the URL frontier.

Choose an explicit URL normalization policy, then store the resulting URL strings as the equality keys. A hash can route or index candidates, but equality must compare the complete normalized strings. A query parameter, case change, or trailing slash must not be discarded unless the policy establishes that it preserves the resource identity needed by this crawl.

Four-byte hashes provide only `2^32 = 4,294,967,296` possible values. They cannot uniquely identify 15 billion distinct URLs. The original size calculation:
```text
        15 Billion URLS  * 4 bytes => 60 GB
```
sizes a collision-prone index, not an exact seen-URL store.

If normalized URLs average `u` bytes, their raw string payload is `15 billion * u` bytes: at an assumed 100 bytes each, 1.5 TB before index overhead and replicas. A shared cache can reduce persistent lookups, but a cache miss does not establish that a URL is new. Approximate membership filters trade memory for false positives; using a positive result as the final decision can skip unseen URLs.

#### h. Checkpointing
A crawl of the entire Web takes weeks to complete. To guard against failures, our crawler can write regular snapshots of its state to the disk. An interrupted on aborted crawl can easily be restarted from the latest checkpoint.


## 6. Fault Tolerance
We should use consistent hashing for distribution among crawler servers. Consistent hashing will help in:
- replacing dead hosts
- distributing load among crawling servers

All crawling servers will perform regular checkpointing and storing their FIFO queues to disks. If a server goes down, we can replace it. During the replacement, consistent hashing should shift the load to other load-capable servers.